# Chapter 2 bounds demo
Numerical sanity checks for FMP bounds on small toy systems.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'mpmgame').exists():
        sys.path.insert(0, str(src_dir))
        break
import numpy as np
import pandas as pd
import mpmgame as mpm
print('using', mpm.__file__)


In [ ]:
G = np.diag([0.8, 0.6, 0.5])
alpha0 = np.array([[0, 0.3, 0], [0.2, 0, 0.1], [0, 0.25, 0]])
system = mpm.build_contract_system(G, alpha0, label='base_static')
Q0 = np.array([[0, 0.4, -0.2], [0.1, 0, 0.3], [-0.1, 0.2, 0]])
lambdas = np.linspace(0, 0.8, 9)
fam_q = mpm.family_vary_q(system, Q0, lambdas, access_model='w2', threat_model='single_link')
fam_q.data.head()


In [ ]:
_ = mpm.plot_vulnerability_vs_parameter(fam_q.data, x='lambda', title='Family A: vary Q', threat_model='single_link')


In [ ]:
rhos = np.linspace(0, 1.0, 11)
fam_a = mpm.family_vary_alpha(G, alpha0, rhos, Q=np.zeros_like(G), access_model='w2', threat_model='full')
_ = mpm.plot_vulnerability_vs_parameter(fam_a.data, x='rho', title='Family B: vary alpha', threat_model='full')


In [ ]:
fam_access = mpm.family_vary_access(system, Q=np.zeros_like(G), access_models=['w1','w2','w3'], threat_model='full')
fam_access.data
_ = mpm.plot_access_model_comparison(fam_access.data, threat_model='full')


In [ ]:
rand = mpm.random_bound_experiments(n_samples=80, n=3, seed=3, access_model='w2', threat_model='single_link')
rand.accepted.head()
if len(rand.accepted):
    _ = mpm.plot_bound_scatter(rand.accepted, bound_col='lower_bound', title='Random systems: measured vs lower bound')
rand.rejected.head()


In [ ]:
summary = mpm.summarize_experiment_results(fam_q, fam_a, fam_access)
summary
mpm.export_report_markdown('reports/chapter2_demo_report.md', summary, headline='Chapter-2 demo report (notebook)')
